# GraphRunner Branching Demo

Shows conditional, parallel, and batched nodes using `GraphRunner` with run_id tracing.

In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (press Enter to skip): ")

from agentic_codex import AgentBuilder, Context, GraphRunner, GraphNodeSpec, FunctionAdapter, EnvOpenAIAdapter
from agentic_codex.core.schemas import AgentStep, Message

def stub_llm(prompt: str) -> str:
    lines = [ln.strip() for ln in prompt.splitlines() if ln.strip()]
    return f"[stub llm] {lines[-1][:120] if lines else 'response'}"

try:
    llm_adapter = EnvOpenAIAdapter(model="gpt-4o-mini")
except Exception:
    llm_adapter = FunctionAdapter(stub_llm)

def make_agent(name: str, tag: str):
    def step(ctx: Context) -> AgentStep:
        payload = ctx.scratch.get("batch_item", tag)
        msg = f"{tag}:{payload}:{ctx.goal}"
        return AgentStep(out_messages=[Message(role=name, content=msg)], state_updates={tag: payload})
    return AgentBuilder(name=name, role=tag).with_llm(llm_adapter).with_step(step).build()

ingest = make_agent("ingest", "ingest")
analyze = make_agent("analyze", "analyze")
review = make_agent("review", "review")
ship = make_agent("ship", "ship")

graph = {
    "ingest": GraphNodeSpec(agent=ingest),
    "analyze": GraphNodeSpec(agent=analyze, after=["ingest"], batch_key="docs", parallel_group="analysis", parallel=True, name="analyze_batch"),
    "review": GraphNodeSpec(agent=review, after=["analyze"], condition=lambda ctx: ctx.scratch.get("review", True)),
    "ship": GraphNodeSpec(agent=ship, after=["review"], condition=lambda ctx: True),
}

ctx = Context(goal="Process docs", scratch={"docs": ["docA", "docB"]})
runner = GraphRunner(graph, allow_parallel=True, isolate_parallel_state=True)
result = runner.run(goal=ctx.goal, inputs=ctx.scratch)
[m.content for m in result.messages]